<a id="setup"></a>
# Expérience CorBaMa — tagset fin, tokenisation mot-entier

Contrairement aux modules précédents, on entraîne ici directement sur CorBaMa, avec son tagset fin (24 catégories linguistiques). Le corpus est petit (~4 640 phrases) comparé à Bayelemabaga (37 392), donc les résultats doivent être lus comme une exploration, pas comme un remplacement du modèle principal.

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

Device : cpu


<a id="split"></a>
# Chargement, tagset fin et split train/val/test de CorBaMa

In [3]:
corbama_fine_path = Path.cwd().parent / "08_real_data_evaluation" / "corbama_corpus_fine.tsv"
corbama_fine_data = load_bambara_corpus(corbama_fine_path)

label2id_fine, id2label_fine = build_fine_tagset(corbama_fine_data)
print(f"Tagset fin détecté : {len(label2id_fine)} classes (incl. <PAD>)")
print(sorted(label2id_fine.keys()))

Corpus chargé : 166361 phrases.
Tagset fin détecté : 32 classes (incl. <PAD>)
['<PAD>', 'PUNCT', 'adj', 'adj/n', 'adv', 'adv.p', 'adv/n', 'conj', 'conj/prep', 'conv.n', 'cop', 'dtm', 'etrg', 'intj', 'mrph', 'n', 'n.prop', 'n/adj', 'n/v', 'num', 'onomat', 'perp', 'pers', 'pm', 'pp', 'prep', 'prn', 'prn/dtm', 'prt', 'ptcp', 'v', 'vq']


In [4]:
# Split propre à CorBaMa (même seed=42 par convention, mais un split différent
# de celui de Bayelemabaga puisque c'est un corpus différent)
corbama_train, corbama_val, corbama_test = split_corpus(corbama_fine_data, seed=42)
print(f"Train : {len(corbama_train)} | Val : {len(corbama_val)} | Test : {len(corbama_test)}")

Train : 133088 | Val : 16636 | Test : 16637


<a id="training"></a>
# Entraînement du modèle mot-entier sur le tagset fin

Le vocabulaire est construit uniquement sur corbama_train, comme partout ailleurs dans le projet.

In [5]:
word_to_ix_cb, _ = build_vocab(corbama_train)
print(f"Taille du vocabulaire (CorBaMa train) : {len(word_to_ix_cb)}")

hyperparams = {
    "learning_rate": 1e-3,
    "batch_size": 16,     # corpus petit -> batch plus petit
    "hidden_dim": 128,
    "embedding_dim": 100,
}

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)
checkpoint_path = MODELS_DIR / "corbama_word_finetagset.pth"

Taille du vocabulaire (CorBaMa train) : 74722


In [6]:
model_word, history_word, best_val_loss_word = run_training(
    corbama_train, corbama_val,
    dataset_cls=BambaraPOSDataset,
    dataset_args=(word_to_ix_cb, label2id_fine),
    vocab_size=len(word_to_ix_cb),
    tagset_size=len(label2id_fine),
    hyperparams=hyperparams,
    device=device,
    patience=5, max_epochs=40,
    checkpoint_path=checkpoint_path,
)

Époque  1 | Train Loss : 0.2402 | Val Loss : 0.1314 | Val Acc : 96.01%
Époque  2 | Train Loss : 0.1047 | Val Loss : 0.1044 | Val Acc : 96.88%
Époque  3 | Train Loss : 0.0757 | Val Loss : 0.0954 | Val Acc : 97.14%
Époque  4 | Train Loss : 0.0584 | Val Loss : 0.0952 | Val Acc : 97.24%
Époque  5 | Train Loss : 0.0465 | Val Loss : 0.0964 | Val Acc : 97.20%
Époque  6 | Train Loss : 0.0375 | Val Loss : 0.0976 | Val Acc : 97.32%
Époque  7 | Train Loss : 0.0305 | Val Loss : 0.1013 | Val Acc : 97.23%
Époque  8 | Train Loss : 0.0251 | Val Loss : 0.1058 | Val Acc : 97.23%
Arrêt anticipé à l'époque 8.


<a id="evaluation"></a>
# Évaluation sur le test set CorBaMa (tagset fin)

In [7]:
def evaluate_word_model_fine(model, word_to_ix, sentences, id2label, device):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for words, tags in sentences:
            ids = [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in words]
            input_tensor = torch.tensor([ids], dtype=torch.long).to(device)
            preds = torch.argmax(model(input_tensor), dim=-1).squeeze(0).cpu().tolist()
            all_true.extend(tags)
            all_pred.extend([id2label[p] for p in preds])
    return all_true, all_pred

true_word, pred_word = evaluate_word_model_fine(model_word, word_to_ix_cb, corbama_test, id2label_fine, device)
acc_word = accuracy_score(true_word, pred_word)

print(f"Accuracy mot-entier, tagset fin, sur test CorBaMa : {acc_word*100:.2f}%\n")
print(classification_report(true_word, pred_word, zero_division=0))

Accuracy mot-entier, tagset fin, sur test CorBaMa : 97.13%

              precision    recall  f1-score   support

       PUNCT       1.00      1.00      1.00     39359
         adj       0.91      0.87      0.89      2692
       adj/n       0.00      0.00      0.00         1
         adv       0.94      0.91      0.93      2301
       adv.p       0.91      0.71      0.80        42
        conj       0.97      0.97      0.97     10312
      conv.n       0.39      0.22      0.28       104
         cop       0.93      0.97      0.95      9844
         dtm       0.92      0.95      0.93      8032
        intj       0.94      0.89      0.92       733
        mrph       0.50      0.50      0.50        12
           n       0.96      0.97      0.96     54604
      n.prop       0.93      0.91      0.92      8787
       n/adj       0.00      0.00      0.00         2
         n/v       0.00      0.00      0.00         6
         num       0.99      0.98      0.98      5348
      onomat       0.

# **Interprétation**

En regardant en particulier les scores sur ***pm vs cop*** : ce sont deux catégories que notre tagset réduit à 11 classes fusionnait toutes les deux dans **AUX**.